# 1. Fireworks AI + LangGraph RAG App



## 1. Environment and Dependencies



In [29]:
# --- Environment: load .env and ensure API key is set ---
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

#if not os.environ.get("FIREWORKS_API_KEY"):
os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Enter your Fireworks API key: ")

# Model endpoints: use .env (FIREWORKS_CHAT_MODEL, FIREWORKS_EMBEDDING_MODEL) or fallbacks
FIREWORKS_CHAT_MODEL = os.environ.get(
    "FIREWORKS_CHAT_MODEL",
    "accounts/fireworks/models/gpt-oss-20b",
)
FIREWORKS_EMBEDDING_MODEL = os.environ.get(
    "FIREWORKS_EMBEDDING_MODEL",
    "accounts/fireworks/models/qwen3-embedding-8b",
)

# --- Imports used across the notebook ---
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langgraph.graph import END, START, StateGraph
from langchain_fireworks import ChatFireworks
from langchain_core.prompts import ChatPromptTemplate
from typing import Annotated, TypedDict
from langchain_core.output_parsers import StrOutputParser


## 2. Load Documents



In [30]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

try:
    directory_loader = DirectoryLoader(
        'data', glob="**/*.pdf", loader_cls=PyMuPDFLoader
    )
    documents = directory_loader.load()
except Exception:
    documents = []

## 3. Build the Vector Store / Retriever



In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
import tiktoken


def _tiktoken_len(text: str) -> int:
    """Return token length using tiktoken; used for chunk length measurement."""
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(text)
    return len(tokens)
    
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=_tiktoken_len
)
chunks = text_splitter.split_documents(documents) if documents else []

# Embeddings and vector store (in-memory Qdrant)
embedding_model = OpenAIEmbeddings(
    model=os.environ.get("FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base="https://api.fireworks.ai/inference/v1",
    check_embedding_ctx_length=False,
    dimensions=4096,
)
qdrant_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    location=":memory:",
    collection_name="rag_collection",
)
retriever = qdrant_vectorstore.as_retriever()

In [32]:
human_template = (
    "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
    "Use the provide context to answer the provided user query. "
    "Only use the provided context to answer the query. If you do not know the answer, or it's not contained in the provided context respond with \"I don't know\""
)
chat_prompt = ChatPromptTemplate.from_messages([("human", human_template)])
generator_llm = ChatOpenAI(
    model=os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base="https://api.fireworks.ai/inference/v1",
)

## 4. LangGraph RAG Workflow



In [33]:
class _RAGState(TypedDict):
    """State schema for the simple two-step RAG graph: retrieve then generate."""
    question: str
    context: list[Document]
    response: str

# LLM used only inside this graph (so RAGAS cells cannot overwrite it)
_fireworks_llm = ChatOpenAI(
    model=os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base="https://api.fireworks.ai/inference/v1",
)

def retrieve(state: _RAGState) -> _RAGState:
    retrieved_docs = retriever.invoke(state["question"]) if retriever else []
    return {"context": retrieved_docs}  # type: ignore

def generate(state: _RAGState) -> _RAGState:
    generator_chain = chat_prompt | _fireworks_llm | StrOutputParser()
    response_text = generator_chain.invoke(
        {"query": state["question"], "context": state.get("context", [])}
    )
    return {"response": response_text}  

graph_builder = StateGraph(_RAGState)
graph_builder = graph_builder.add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
custom_graph = graph_builder.compile()

# OpenAI RAG

In [12]:
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

In [41]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data",
    embedding=embeddings,
)
_ = vector_store.add_documents(documents=chunks)

retriever_openai = vector_store.as_retriever()

In [42]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")
def retrieve(state: _RAGState) -> _RAGState:
    retrieved_docs = retriever_openai.invoke(state["question"]) if retriever_openai else []
    return {"context": retrieved_docs} 

def generate(state: _RAGState) -> _RAGState:
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = chat_prompt.format_messages(query=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

graph_builder = StateGraph(_RAGState)
graph_builder = graph_builder.add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
openai_graph = graph_builder.compile()

# RAGAS & Langsmith

In [15]:

import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
if not os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key (optional, for tracing):")

In [18]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

testset_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
generator = TestsetGenerator(llm=testset_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(documents, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/22 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Property 'summary' already exists in node '6d2fce'. Skipping!
Property 'summary' already exists in node '3ca025'. Skipping!
Property 'summary' already exists in node '8f11ea'. Skipping!
Property 'summary' already exists in node '323594'. Skipping!


...

Property 'summary_embedding' already exists in node '6d2fce'. Skipping!
Property 'summary_embedding' already exists in node '323594'. Skipping!
Property 'summary_embedding' already exists in node '3ca025'. Skipping!
Property 'summary_embedding' already exists in node '8f11ea'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [43]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import ContextRecall, ContextEntityRecall, ContextPrecision, Faithfulness, AnswerRelevancy, AnswerCorrectness
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

custom_run_config = RunConfig(timeout=360)

retrievers = {
    "fireworks": custom_graph,
    "openai": openai_graph,
}

for test_row in dataset:
    response = custom_graph.invoke({"question" : test_row.eval_sample.user_input})
    msg = response["response"]
    test_row.eval_sample.response = msg.content if hasattr(msg, "content") else str(msg)
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())
result = evaluate(
dataset=evaluation_dataset,
metrics=[ContextRecall(), ContextEntityRecall(), ContextPrecision(), Faithfulness(), AnswerRelevancy(), AnswerCorrectness()],
llm=evaluator_llm,
run_config=custom_run_config
)

result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10863d600> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10863d600> is already entered


{'context_recall': 0.9042, 'context_entity_recall': 0.3192, 'context_precision': 0.8935, 'faithfulness': 0.5611, 'answer_relevancy': 0.6115, 'answer_correctness': 0.5011}

In [44]:
for test_row in dataset:
    response = openai_graph.invoke({"question" : test_row.eval_sample.user_input})
    msg = response["response"]
    test_row.eval_sample.response = msg.content if hasattr(msg, "content") else str(msg)
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())
result = evaluate(
dataset=evaluation_dataset,
metrics=[ContextRecall(), ContextEntityRecall(), ContextPrecision(), Faithfulness(), AnswerRelevancy(), AnswerCorrectness()],
llm=evaluator_llm,
run_config=custom_run_config
)

result

Task was destroyed but it is pending!
task: <Task pending name='Task-7767' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/karlaudiljak/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-7768' coro=<Kernel.shell_main() running at /Users/karlaudiljak/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/karlaudiljak/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/http/cookiejar.py:1227: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def deepvalues(mapping):
Task was destroyed but it is pending!
task: <Task pending name='Task-7768' coro=<Kernel.shell_main() running at /Users/karlaudiljak/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packag

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10863d600> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10863d600> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-7875' coro=<_async_in_context.<locals>.

{'context_recall': 1.0000, 'context_entity_recall': 0.2283, 'context_precision': 0.9606, 'faithfulness': 0.9638, 'answer_relevancy': 0.9439, 'answer_correctness': 0.6701}

# Langsmith

In [46]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"llm_rag_dataset"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

In [47]:
from langsmith import evaluate

evaluate(
    custom_graph.invoke,
    data=dataset_name,
    metadata={"revision_id": "custom_graph"},
)

View the evaluation results for experiment: 'weary-sound-5' at:
https://smith.langchain.com/o/f4f61912-4210-4d7a-8df0-599526ce1676/datasets/01e80a5a-53bf-4099-88d1-20de3dd9cf12/compare?selectedSessions=2d21b8b2-2689-496a-9be5-b58f8c05573a




0it [00:00, ?it/s]

Error running target function: Error code: 429 - {'error': {'message': 'You have exceeded your rate limit for this API. Please try again later. For more information, see https://docs.fireworks.ai/guides/quotas_usage/rate-limits.', 'param': None, 'code': 'RATE_LIMIT_EXCEEDED', 'type': 'error'}, 'request_id': 'embd-dc92867d8b2049b2a00e23c23e8b1bdd'}
Traceback (most recent call last):
  File "/Users/karlaudiljak/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packages/langsmith/evaluation/_runner.py", line 1903, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/karlaudiljak/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packages/langgraph/pregel/main.py", line 3094, in invoke
    for chunk in self.stream(
                 ~~~~~~~~~~~^
        input,
        ^^^^^^
    ...<10 lines>...
        **kwargs,
        ^^^^^^^^^
    ):
    ^
  File "/Users/karlaudiljak/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packages/langgra

,inputs.question,outputs.question,outputs.context,outputs.response,error,reference.answer,execution_time,example_id,id,outputs.output
0,How can veterinarians use annual examinations ...,How can veterinarians use annual examinations ...,"[page_content='For example, some senior cats a...",According to the 2021 AAHA/AAFP Feline Life‑St...,NaN,Veterinarians can use annual examinations as a...,7.891923,8680b81d-27e4-4aa3-b680-9dc7e8d0cc85,019cd9c7-5c35-79c1-98a4-89bf96a2be78,NaN
1,Acccording to the 2021 AAHA/AAFP Feline Life S...,Acccording to the 2021 AAHA/AAFP Feline Life S...,"[page_content='Currently, in North America, op...",Positive reinforcement is a cornerstone of kit...,NaN,The 2021 AAHA/AAFP Feline Life Stage Guideline...,8.384622,1d7c329f-8519-42f9-86fa-be38365efc2c,019cd9c7-7b0a-7c02-8ef9-56886ed044cd,NaN
2,"According to the AAFP Position Statement, what...","According to the AAFP Position Statement, what...","[page_content='Currently, in North America, op...",**AAFP Position Statement recommendations**\n\...,NaN,The AAFP Position Statement recommends that ca...,9.763298,fc1b59f3-7812-487a-92e8-b38d3bf9f55f,019cd9c7-9bcc-76d2-9acb-bf8f710e15e8,NaN
3,According to the 2019 AAFP Feline Zoonoses Gui...,According to the 2019 AAFP Feline Zoonoses Gui...,[page_content='thorough and accurate preventiv...,**Key preventive‑health strategies from the 20...,NaN,The 2019 AAFP Feline Zoonoses Guidelines recom...,15.363409,63d61900-06f7-45aa-8cc7-fe3213aa95b0,019cd9c7-c1f0-7f61-b6dc-f3db8112f6b3,NaN
4,According to the 2021 AAHA/AAFP Feline Life St...,According to the 2021 AAHA/AAFP Feline Life St...,[page_content='VETERINARY PRACTICE GUIDELINES\...,**Key considerations for aging and geriatric c...,NaN,The 2021 AAHA/AAFP Feline Life Stage Guideline...,12.565869,e5a9f265-8698-47bd-9414-fc1fda8216a3,019cd9c7-fdf5-7902-a20d-080bcb5326cc,NaN
5,How can environmental enrichment be adapted to...,NaN,NaN,NaN,"RateLimitError(""Error code: 429 - {'error': {'...",Environmental enrichment for senior cats shoul...,1.725453,2079cbba-1072-434b-8230-fa96b594940b,019cd9c8-2f0c-7560-b997-4be546e375e8,NaN
6,How can routine healthcare examinations help w...,How can routine healthcare examinations help w...,[page_content='detection of changes and identi...,Routine health‑check visits give veterinarians...,NaN,Routine healthcare examinations are essential ...,12.453258,77bcc0c9-b404-418a-9aac-7a9656184138,019cd9c8-35cd-7c23-b329-373f0f7f09ae,NaN
7,How can kitten socialization during the sensit...,How can kitten socialization during the sensit...,"[page_content='Currently, in North America, op...",Kitten socialization and positive‑reinforcemen...,NaN,Kitten socialization during the sensitive peri...,11.830373,94eb7aeb-ffc0-4eff-bb83-e6f921d1505c,019cd9c8-6674-7782-bb69-8b453639638b,NaN
8,What topics are included in Table 2 that are r...,What topics are included in Table 2 that are r...,[page_content='TABLE 2 (Continued)\n2021 AAHA/...,I don't know.,NaN,Table 2 lists discussion items relevant to all...,3.312073,3d92f70f-be76-4b6b-9922-365ec99998b6,019cd9c8-94ab-75b0-993d-66faffc2e14e,NaN
9,Wut shud I kno abot oldr cats?,Wut shud I kno abot oldr cats?,[page_content='TABLE 2 (Continued)\n2021 AAHA/...,**What you should know about senior (older) ca...,NaN,Some senior cats aged 10 years and older may r...,6.768875,aafd2f5a-ca04-495d-a652-8fb5e44a4a41,019cd9c8-a19c-7dc2-bb1c-71671577f9c1,NaN


In [48]:
evaluate(
    openai_graph.invoke,
    data=dataset_name,
    metadata={"revision_id": "openai_graph"},
)

View the evaluation results for experiment: 'clear-land-88' at:
https://smith.langchain.com/o/f4f61912-4210-4d7a-8df0-599526ce1676/datasets/01e80a5a-53bf-4099-88d1-20de3dd9cf12/compare?selectedSessions=d2c5e40a-de22-4c40-9213-923dce36a75b




0it [00:00, ?it/s]

,inputs.question,outputs.question,outputs.context,outputs.response,error,reference.answer,execution_time,example_id,id
0,How can veterinarians use annual examinations ...,How can veterinarians use annual examinations ...,"[page_content='For example, some senior cats a...","According to the guidelines, veterinarians can...",None,Veterinarians can use annual examinations as a...,3.422149,8680b81d-27e4-4aa3-b680-9dc7e8d0cc85,019cd9c8-dc93-7cc1-90fa-fe0e2621c5a6
1,Acccording to the 2021 AAHA/AAFP Feline Life S...,Acccording to the 2021 AAHA/AAFP Feline Life S...,"[page_content='Currently, in North America, op...",According to the 2021 AAHA/AAFP Feline Life St...,None,The 2021 AAHA/AAFP Feline Life Stage Guideline...,8.453750,1d7c329f-8519-42f9-86fa-be38365efc2c,019cd9c8-e9f2-79c3-b3e9-809966a15209
2,"According to the AAFP Position Statement, what...","According to the AAFP Position Statement, what...",[page_content='Conﬂict may occur when a new ca...,According to the AAFP Position Statement refer...,None,The AAFP Position Statement recommends that ca...,2.670926,fc1b59f3-7812-487a-92e8-b38d3bf9f55f,019cd9c9-0af9-7e22-abed-8cc694328299
3,According to the 2019 AAFP Feline Zoonoses Gui...,According to the 2019 AAFP Feline Zoonoses Gui...,"[page_content='space, administration at this l...",According to the 2019 AAFP Feline Zoonoses Gui...,None,The 2019 AAFP Feline Zoonoses Guidelines recom...,5.215578,63d61900-06f7-45aa-8cc7-fe3213aa95b0,019cd9c9-1569-7eb1-9add-aa58110f393a
4,According to the 2021 AAHA/AAFP Feline Life St...,According to the 2021 AAHA/AAFP Feline Life St...,"[page_content='90. Wichert B, Muller L, Gebert...",According to the 2021 AAHA/AAFP Feline Life St...,None,The 2021 AAHA/AAFP Feline Life Stage Guideline...,6.435047,e5a9f265-8698-47bd-9414-fc1fda8216a3,019cd9c9-29c9-7040-9b81-b7a261071225
5,How can environmental enrichment be adapted to...,How can environmental enrichment be adapted to...,"[page_content='including carpeting, window and...",Environmental enrichment for senior cats shoul...,None,Environmental enrichment for senior cats shoul...,4.009853,2079cbba-1072-434b-8230-fa96b594940b,019cd9c9-42ed-7bb0-9d99-2ff441edf677
6,How can routine healthcare examinations help w...,How can routine healthcare examinations help w...,[page_content='detection of changes and identi...,Routine healthcare examinations are crucial fo...,None,Routine healthcare examinations are essential ...,6.458849,77bcc0c9-b404-418a-9aac-7a9656184138,019cd9c9-5298-75b2-8123-a507786cc7a4
7,How can kitten socialization during the sensit...,How can kitten socialization during the sensit...,"[page_content='Currently, in North America, op...",Kitten socialization during the sensitive peri...,None,Kitten socialization during the sensitive peri...,12.123085,94eb7aeb-ffc0-4eff-bb83-e6f921d1505c,019cd9c9-6bd4-7113-bd47-7b35c8c9466a
8,What topics are included in Table 2 that are r...,What topics are included in Table 2 that are r...,"[page_content='For example, some senior cats a...",The topics included in Table 2 that are releva...,None,Table 2 lists discussion items relevant to all...,2.238893,3d92f70f-be76-4b6b-9922-365ec99998b6,019cd9c9-9b31-7ea1-a2b4-6cb2c0612da8
9,Wut shud I kno abot oldr cats?,Wut shud I kno abot oldr cats?,[page_content='detection of changes and identi...,"For older cats (mature adult and senior cats),...",None,Some senior cats aged 10 years and older may r...,7.268559,aafd2f5a-ca04-495d-a652-8fb5e44a4a41,019cd9c9-a3f1-7333-93c1-8a085da2bb64


# LangSmith: Fireworks vs OpenAI — Latency, Token Count & Cost

The dashboard below compares the **Fireworks** RAG pipeline (weary-sound-5) and the **OpenAI** RAG pipeline (clear-land-88) on the same dataset.

![LangSmith comparison: Fireworks vs OpenAI](langsmith_fireworks_vs_openai.png)

### Comparison summary

| Metric | Fireworks (weary-sound-5) | OpenAI (clear-land-88) |
|--------|---------------------------|------------------------|
| **P50 latency** | ~8 | ~4.5 |
| **P99 latency** | ~15 | ~11.5 |
| **Input tokens** | ~36K | ~33K |
| **Output tokens** | ~9K | ~3K |
| **Total tokens** | ~45K | ~36K |
| **Input cost** | ~$0.0048 | ~$0.0060 |
| **Output cost** | ~$0.0052 | ~$0.0045 |
| **Total cost** | ~$0.0100 | ~$0.0105 |

- **Latency:** OpenAI has lower P50 and P99 latency than Fireworks for this run, so responses start and complete faster with the OpenAI pipeline.

- **Token count:** Fireworks used more total tokens (~45K vs ~36K), mainly due to higher output tokens (~9K vs ~3K). Input token usage is similar (~36K vs ~33K).

- **Cost:** Total cost is similar for both (~$0.01). Fireworks has lower input cost but higher output cost; OpenAI has higher input cost and lower output cost, so the totals are close.

### RAGAS metrics

| Metric | Fireworks (weary-sound-5) | OpenAI (clear-land-88) |
|--------|---------------------------|------------------------|
| **context_recall** | 0.9042 | 1.0000 |
| **context_entity_recall** | 0.3192 | 0.2283 |
| **context_precision** | 0.8935 | 0.9606 |
| **faithfulness** | 0.5611 | 0.9638 |
| **answer_relevancy** | 0.6115 | 0.9439 |
| **answer_correctness** | 0.5011 | 0.6701 |

- **Retrieval (context_*):** Fireworks has slightly higher **context_entity_recall** (0.32 vs 0.23); OpenAI has perfect **context_recall** (1.0) and higher **context_precision** (0.96 vs 0.89).

- **Generation quality:** OpenAI scores higher on **faithfulness** (0.96 vs 0.56), **answer_relevancy** (0.94 vs 0.61), and **answer_correctness** (0.67 vs 0.50), so answers stay closer to the context and are more relevant and correct.

- **Summary:** OpenAI outperforms Fireworks on RAGAS in this run, especially on faithfulness and answer quality; Fireworks does slightly better on entity recall.